# Experiment 3: topology rewiring

**Question.** Keep the rule (Conway's B3/S23) and the number of neighbours (8) fixed, and change only *who* is whose neighbour. Starting from a 64 × 64 torus, a fraction $\beta$ of the edges is rewired at random: $\beta = 0$ is the lattice, small $\beta$ gives a *small-world* network, and $\beta = 1$ a random 8-regular network. How do structure, transients, final states and damage spreading change with $\beta$?

Rewiring uses degree-preserving double-edge swaps (`life.networks.rewire`), so every node keeps exactly 8 neighbours. Classic Watts–Strogatz rewiring, which lets degrees vary, is run as a comparison in part 3d.

| Symbol | Meaning |
|---|---|
| $\ell$ | mean shortest-path length: how many hops apart two nodes are on average |
| $C$ | clustering: the chance that two neighbours of a node are also neighbours of each other |

## Predictions (written before any run)

| # | Question | Prediction |
|---|---|---|
| 1 | Which falls first as $\beta$ grows, $\ell$ or $C$? At what $\beta$ is $\ell$ halved? | $C$ falls first; $\ell$ is halved around $\beta \approx 0.1$ |
| 2 | Transient at $\beta = 1$ vs. the lattice (median ≈ 1000) | longer, by about 100 generations |
| 3 | Runs that do not settle within 20 000 generations? | yes; unsure at which $\beta$ |
| 4 | What is left at $\beta = 1$? | still patterns |
| 5 | Period-$4L$ gliders at $\beta = 0.01$? Most common period at $\beta = 1$? | no gliders; period about 100 |
| 6 | Damage on a small world ($\beta = 0.03$) vs. the lattice; same-attractor fraction | spreads more slowly; fraction unchanged |
| 7 | An abrupt change in the dynamics? Does it line up with the fall of $\ell$ or of $C$? | yes, with the fall of $C$ |

In [ ]:
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from life.metrics import damage_spread, detect_cycle, flip
from life.networks import clustering, mean_path_length, rewire, watts_strogatz
from life.rules import CONWAY
from life.topology import Graph, lattice_adjacency

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
FIGURES = ROOT / "docs" / "figures"
DATA = ROOT / "data" / "exp03"
FIGURES.mkdir(parents=True, exist_ok=True)
DATA.mkdir(parents=True, exist_ok=True)

SIZE = 64
N = SIZE * SIZE
DENSITY = 0.35
MAX_STEPS = 20_000
RUNS = 50
BETAS = [0.0, 0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1.0]
LATTICE = lattice_adjacency((SIZE, SIZE), "torus")


def network(beta, seed, method="rewire"):
    rng = np.random.default_rng([int(beta * 1e6), seed, 7])
    build = rewire if method == "rewire" else watts_strogatz
    return build(LATTICE, beta, rng)


def soup(seed):
    return np.random.default_rng(20_000 + seed).random((SIZE, SIZE)) < DENSITY


def final_density(cycle):
    return np.unpackbits(np.frombuffer(cycle.attractor, dtype=np.uint8))[:N].mean()

## 3a. Structure: clustering and path length

Five networks per $\beta$; $\ell$ is estimated from 200 random source nodes. Values are shown relative to the lattice ($\beta = 0$).

In [ ]:
structure = []
for beta in BETAS:
    for seed in range(5):
        a = network(beta, seed)
        rng = np.random.default_rng(seed)
        structure.append((beta, seed, clustering(a), mean_path_length(a, 200, rng)))
structure = np.array(structure)
np.savetxt(
    DATA / "structure.csv",
    structure,
    delimiter=",",
    fmt="%.6g",
    header="beta,seed,clustering,path_length",
    comments="",
)

c0 = structure[structure[:, 0] == 0, 2].mean()
l0 = structure[structure[:, 0] == 0, 3].mean()
print(f"lattice: C = {c0:.3f}, l = {l0:.2f}")
print(f"{'beta':>6} {'C/C0':>7} {'l/l0':>7}")
for beta in BETAS:
    rows = structure[structure[:, 0] == beta]
    print(f"{beta:>6} {rows[:, 2].mean() / c0:>7.3f} {rows[:, 3].mean() / l0:>7.3f}")

In [ ]:
betas_pos = np.array(BETAS[1:])
c_rel = [structure[structure[:, 0] == b, 2].mean() / c0 for b in betas_pos]
l_rel = [structure[structure[:, 0] == b, 3].mean() / l0 for b in betas_pos]

fig, ax = plt.subplots(figsize=(6, 3.8))
ax.semilogx(betas_pos, c_rel, "o-", label="clustering C(β) / C(0)")
ax.semilogx(betas_pos, l_rel, "s-", label="path length ℓ(β) / ℓ(0)")
ax.set_xlabel("rewired fraction β")
ax.set_ylim(0, 1.05)
ax.set_title("Structure of the rewired lattice")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES / "exp03_structure.png", dpi=150)
plt.show()

## 3b. Dynamics: transients, periods, final states

For each $\beta$: 50 runs, each on a freshly rewired network with its own random soup (density 0.35), followed to its cycle (cap 20 000 generations). The final live density is read from a state on the cycle.

In [ ]:
records = []  # beta, seed, transient, period, final density; -1 marks a censored run
for beta in BETAS:
    start = time.perf_counter()
    for seed in range(RUNS):
        cycle = detect_cycle(soup(seed), CONWAY, Graph(network(beta, seed)), MAX_STEPS)
        if cycle is None:
            records.append((beta, seed, -1, -1, -1.0))
        else:
            records.append((beta, seed, cycle.transient, cycle.period, final_density(cycle)))
    print(f"beta = {beta:<6}: {RUNS} runs in {time.perf_counter() - start:.1f} s")

dyn = np.array(records)
np.savetxt(
    DATA / "dynamics.csv",
    dyn,
    delimiter=",",
    fmt="%.6g",
    header="beta,seed,transient,period,final_density",
    comments="",
)

In [ ]:
def summary(table, beta):
    rows = table[table[:, 0] == beta]
    done = rows[rows[:, 2] >= 0]
    censored = int(np.sum(rows[:, 2] < 0))
    if len(done) == 0:
        return censored, np.nan, np.nan, np.nan, ""
    periods, counts = np.unique(done[:, 3].astype(int), return_counts=True)
    order = np.argsort(-counts)[:4]
    top = ", ".join(f"{periods[i]}×{counts[i]}" for i in order)
    return (
        censored,
        np.median(done[:, 2]),
        np.percentile(done[:, 2], 90),
        np.median(done[:, 4]),
        top,
    )


print(f"{'beta':>6} {'cens.':>6} {'median T':>9} {'p90 T':>7} {'density':>8}  periods")
for beta in BETAS:
    cens, med, p90, dens, top = summary(dyn, beta)
    print(f"{beta:>6} {cens:>6} {med:>9.0f} {p90:>7.0f} {dens:>8.3f}  {top}")

In [ ]:
x = np.array(BETAS[1:])
stats = np.array([summary(dyn, b)[:4] for b in BETAS], dtype=float)
lattice = stats[0]

fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
panels = [
    (1, "median transient"),
    (3, "median final live density"),
    (0, "runs not settled (of 50)"),
]
for ax, (col, label) in zip(axes, panels, strict=True):
    ax.semilogx(x, stats[1:, col], "o-", color="#2f7d4f")
    ax.axhline(lattice[col], color="grey", ls="--", label="lattice (β = 0)")
    ax.set_xlabel("rewired fraction β")
    ax.set_title(label)
axes[0].legend()
fig.tight_layout()
fig.savefig(FIGURES / "exp03_dynamics.png", dpi=150)
plt.show()

## 3c. Damage spreading on three topologies

50 pairs per $\beta$ (lattice, small world, random), one random cell flipped, 1500 generations. Each pair runs on its own network. For pairs that settle, the final attractors are compared.

In [ ]:
DAMAGE_BETAS = [0.0, 0.03, 1.0]
DAMAGE_RUNS = 50
DAMAGE_STEPS = 1500

damage = {}
attractors = {}
for beta in DAMAGE_BETAS:
    start = time.perf_counter()
    curves, same, differ, censored = [], 0, 0, 0
    for seed in range(DAMAGE_RUNS):
        g = Graph(network(beta, 100 + seed))
        s = soup(100 + seed)
        cell = tuple(int(v) for v in np.random.default_rng(30_000 + seed).integers(0, SIZE, 2))
        curves.append(damage_spread(s, cell, DAMAGE_STEPS, CONWAY, g))
        a = detect_cycle(s, CONWAY, g, MAX_STEPS)
        b = detect_cycle(flip(s, cell), CONWAY, g, MAX_STEPS)
        if a is None or b is None:
            censored += 1
        elif a.attractor == b.attractor:
            same += 1
        else:
            differ += 1
    damage[f"beta_{beta}"] = np.array(curves)
    attractors[beta] = (same, differ, censored)
    print(f"beta = {beta:<5}: {time.perf_counter() - start:.1f} s")

np.savez_compressed(DATA / "damage.npz", **damage)
print()
print(f"{'beta':>6} {'healed':>7} {'peak d/N':>9} {'at t':>5} {'end d/N':>8}  same/differ/censored")
for beta in DAMAGE_BETAS:
    d = damage[f"beta_{beta}"]
    med = np.median(d, axis=0) / N
    same, differ, censored = attractors[beta]
    print(
        f"{beta:>6} {np.mean(d[:, -1] == 0):>7.0%} {med.max():>9.3f} {med.argmax():>5} "
        f"{med[-1]:>8.3f}  {same}/{differ}/{censored}"
    )

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.8))
t = np.arange(DAMAGE_STEPS + 1)
for beta, color in zip(DAMAGE_BETAS, ["#2f7d4f", "#d1495b", "#3b6ea8"], strict=True):
    d = damage[f"beta_{beta}"] / N
    ax.plot(t, np.median(d, axis=0), color=color, lw=2, label=f"β = {beta} (median)")
    ax.fill_between(t, *np.percentile(d, [25, 75], axis=0), color=color, alpha=0.15)
ax.set_xscale("symlog", linthresh=10)
ax.set_xlabel("generation t")
ax.set_ylabel("damage d(t) / number of cells")
ax.set_title("One flipped cell on three topologies (median and quartiles)")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES / "exp03_damage.png", dpi=150)
plt.show()

## 3d. Comparison: classic Watts–Strogatz (degrees not preserved)

Same soups and seeds as 3b, but rewired with `watts_strogatz`, so nodes end up with different numbers of neighbours.

In [ ]:
WS_BETAS = [0.01, 0.1, 1.0]
ws = []
for beta in WS_BETAS:
    start = time.perf_counter()
    for seed in range(RUNS):
        g = Graph(network(beta, seed, method="ws"))
        cycle = detect_cycle(soup(seed), CONWAY, g, MAX_STEPS)
        if cycle is None:
            ws.append((beta, seed, -1, -1, -1.0))
        else:
            ws.append((beta, seed, cycle.transient, cycle.period, final_density(cycle)))
    print(f"WS beta = {beta:<5}: {time.perf_counter() - start:.1f} s")
ws = np.array(ws)
np.savetxt(
    DATA / "watts_strogatz.csv",
    ws,
    delimiter=",",
    fmt="%.6g",
    header="beta,seed,transient,period,final_density",
    comments="",
)

print()
print(f"{'beta':>6} {'method':>18} {'cens.':>6} {'median T':>9} {'density':>8}  periods")
for beta in WS_BETAS:
    for name, table in [("degree-preserving", dyn), ("Watts-Strogatz", ws)]:
        cens, med, _, dens, top = summary(table, beta)
        print(f"{beta:>6} {name:>18} {cens:>6} {med:>9.0f} {dens:>8.3f}  {top}")

## Side by side: the same soup on the lattice and on a small world

Left: $\beta = 0$. Right: $\beta = 0.01$ (1 % of edges rewired). Cells are drawn at their lattice positions in both panels; on the right, a few of them have neighbours far away.

In [ ]:
from matplotlib.animation import FuncAnimation, PillowWriter
from matplotlib.colors import ListedColormap

from life.engine import run

frames = 300
start_state = soup(0)
left = run(start_state, frames, CONWAY, Graph(LATTICE))
right = run(start_state, frames, CONWAY, Graph(network(0.01, 0)))

cmap = ListedColormap(["#f4f5f6", "#2f7d4f"])
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(5.2, 2.9), dpi=80)
im1 = ax1.imshow(left[0], cmap=cmap, vmin=0, vmax=1)
im2 = ax2.imshow(right[0], cmap=cmap, vmin=0, vmax=1)
for ax in (ax1, ax2):
    ax.set_xticks([])
    ax.set_yticks([])
t1 = ax1.set_title(f"lattice · t = {frames}", fontsize=8)
t2 = ax2.set_title(f"β = 0.01 · t = {frames}", fontsize=8)
fig.tight_layout()


def update(i):
    im1.set_data(left[i])
    im2.set_data(right[i])
    t1.set_text(f"lattice · t = {i}")
    t2.set_text(f"β = 0.01 · t = {i}")
    return im1, im2, t1, t2


animation = FuncAnimation(fig, update, frames=range(0, frames + 1, 3))
out = FIGURES / "exp03_side_by_side.gif"
animation.save(out, writer=PillowWriter(fps=12))
plt.close(fig)
print(f"saved {out.name}: {out.stat().st_size / 1e6:.2f} MB")

## Results vs. predictions

*To be written after looking at the output above.*